<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/solved/12_robust_student_t_and_loo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 12 — Robust likelihood and predictive comparison

Return to the Gaussian hierarchical models, use leave-one-out predictive checks to expose influential observations, and compare them with Student-t observation models.

## Setup

This course pins PyMC, modular ArviZ, and Bambi for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1" \
    "bambi==0.21.0"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bambi as bmb
import pymc as pm
import arviz_base as azb
import arviz_stats as azs
import arviz_plots as azp

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("Bambi:", bmb.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

In [ ]:
def diagnostic_overview(idata):
    """Compact sampling diagnostics used throughout the notebook sequence."""
    divergences = int(idata["sample_stats"]["diverging"].sum().item())
    rhat = idata["posterior"].azstats.rhat().to_array()
    ess = idata["posterior"].azstats.ess(method="bulk").to_array()
    print("Divergences:", divergences)
    print("Worst R-hat:", float(rhat.max(skipna=True).item()))
    print("Smallest bulk ESS:", float(ess.min(skipna=True).item()))


def hdi_bounds(draws, prob):
    hdi = azs.hdi(draws, prob=prob)
    return (
        hdi.sel(ci_bound="lower").to_numpy(),
        hdi.sel(ci_bound="upper").to_numpy(),
    )


def expected_response_draws(prediction, family):
    """Expected response on the reaction-time scale for the families used here."""
    posterior = prediction["posterior"]
    if family in {"gaussian", "t"}:
        return posterior["mu"]
    if family == "lognormal":
        return np.exp(posterior["mu"] + 0.5 * posterior["sigma"] ** 2)
    if family == "exgaussian":
        return posterior["mu"] + posterior["nu"]
    raise ValueError(f"Unsupported family: {family}")


def plot_population_fit(model, idata, family="gaussian", title="Population-average relationship"):
    """Posterior mean relationship, excluding participant-specific deviations."""
    grid = pd.DataFrame({"Days": np.linspace(0, 7, 100)})
    pred = model.predict(
        idata,
        kind="response_params",
        data=grid,
        inplace=False,
        include_group_specific=False,
        random_seed=RANDOM_SEED,
    )
    draws = expected_response_draws(pred, family)
    mean = draws.mean(("chain", "draw")).to_numpy()
    lo50, hi50 = hdi_bounds(draws, 0.50)
    lo90, hi90 = hdi_bounds(draws, 0.90)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(sleep["Days"], sleep["Reaction"], s=16, color="black", alpha=0.32)
    ax.fill_between(grid["Days"], lo90, hi90, alpha=0.25, label="90% HDI")
    ax.fill_between(grid["Days"], lo50, hi50, alpha=0.45, label="50% HDI")
    ax.plot(grid["Days"], mean, color="C1", lw=2, label="mean")
    ax.set(xlabel="Days of sleep deprivation", ylabel="Reaction time (ms)", title=title)
    ax.legend(frameon=False)
    plt.show()


def plot_subject_fits(model, idata, family="gaussian", title="Participant trajectories"):
    """Posterior mean relationship for each observed participant."""
    subjects = sleep["Subject"].drop_duplicates().tolist()
    day_grid = np.linspace(0, 7, 50)
    grid = pd.DataFrame(
        [(subject, day) for subject in subjects for day in day_grid],
        columns=["Subject", "Days"],
    )
    pred = model.predict(
        idata,
        kind="response_params",
        data=grid,
        inplace=False,
        include_group_specific=True,
        random_seed=RANDOM_SEED,
    )
    draws = expected_response_draws(pred, family)
    mean = draws.mean(("chain", "draw")).to_numpy().reshape(len(subjects), len(day_grid))
    lo90, hi90 = hdi_bounds(draws, 0.90)
    lo90 = lo90.reshape(len(subjects), len(day_grid))
    hi90 = hi90.reshape(len(subjects), len(day_grid))

    fig, axes = plt.subplots(3, 6, figsize=(12, 7), sharex=True, sharey=True)
    for i, (ax, subject) in enumerate(zip(axes.ravel(), subjects)):
        observed = sleep[sleep["Subject"] == subject]
        ax.scatter(observed["Days"], observed["Reaction"], s=15, color="black", zorder=3)
        ax.fill_between(day_grid, lo90[i], hi90[i], alpha=0.20)
        ax.plot(day_grid, mean[i], color="C1", lw=1.5)
        ax.set_title(f"Subject {subject}", fontsize=9)
    fig.supxlabel("Days of sleep deprivation")
    fig.supylabel("Reaction time (ms)")
    fig.suptitle(title, y=1.01)
    fig.tight_layout()
    plt.show()

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

That zero point is scientifically meaningful, so every Bambi model in this sequence uses `center_predictors=False`. The `Intercept` prior is therefore a prior on baseline reaction time rather than reaction time at the average deprivation day.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

print(f"{sleep['Subject'].nunique()} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

# 12.1 Difficult-to-predict observations

Do Gaussian hierarchical models contain observations that are difficult to predict when they are left out of the fit?

We refit four standalone models here so the notebook does not depend on state from earlier notebooks: Gaussian and Student-t likelihoods, each with varying intercepts only or varying intercepts plus slopes. Bambi's group-specific effects are independent, as in notebooks 5–11.

In [ ]:
base_vi_priors = {
    "Intercept": bmb.Prior("Normal", mu=250, sigma=100),
    "Days": bmb.Prior("Normal", mu=0, sigma=20),
    "sigma": bmb.Prior("Exponential", lam=0.04),
    "1|Subject": bmb.Prior(
        "Normal", mu=0, sigma=bmb.Prior("Exponential", lam=0.04)
    ),
}
base_vis_priors = {
    **base_vi_priors,
    "Days|Subject": bmb.Prior(
        "Normal", mu=0, sigma=bmb.Prior("Exponential", lam=0.10)
    ),
}

models = {
    "gaussian_intercept": bmb.Model(
        "Reaction ~ Days + (1 | Subject)", sleep, family="gaussian",
        priors=base_vi_priors, categorical="Subject", center_predictors=False,
    ),
    "gaussian_slope": bmb.Model(
        "Reaction ~ Days + (1 + Days | Subject)", sleep, family="gaussian",
        priors=base_vis_priors, categorical="Subject", center_predictors=False,
    ),
    "student_t_intercept": bmb.Model(
        "Reaction ~ Days + (1 | Subject)", sleep, family="t",
        priors=base_vi_priors, categorical="Subject", center_predictors=False,
    ),
    "student_t_slope": bmb.Model(
        "Reaction ~ Days + (1 + Days | Subject)", sleep, family="t",
        priors=base_vis_priors, categorical="Subject", center_predictors=False,
    ),
}
models["student_t_slope"]

In [ ]:
idatas = {}
for name, comparison_model in models.items():
    print("\n---", name, "---")
    idatas[name] = comparison_model.fit(
        draws=1000, tune=1500, chains=4, target_accept=0.95, random_seed=RANDOM_SEED
    )
    diagnostic_overview(idatas[name])
    comparison_model.predict(
        idatas[name], kind="response", inplace=True, random_seed=RANDOM_SEED
    )
    comparison_model.compute_log_likelihood(idatas[name])

These intervals use PSIS-LOO weighting rather than the ordinary full-data posterior predictive distribution. Large discrepancies help identify observations for which the fitted model depends strongly on seeing that observation.

In [ ]:
azp.plot_loo_interval(
    idatas["gaussian_slope"], var_names=["Reaction"],
    point_estimate="mean", ci_probs=(0.50, 0.90), ci_kind="hdi",
    figure_kwargs={"figsize": (11, 4)},
);

In [ ]:
azp.plot_loo_interval(
    idatas["student_t_slope"], var_names=["Reaction"],
    point_estimate="mean", ci_probs=(0.50, 0.90), ci_kind="hdi",
    figure_kwargs={"figsize": (11, 4)},
);

# 12.2 Robust predictive calibration

Does replacing the Gaussian likelihood with a Student-t likelihood improve leave-one-out calibration?

In [ ]:
azp.plot_loo_pit(idatas["gaussian_slope"], var_names=["Reaction"]);
azp.plot_loo_pit(idatas["student_t_slope"], var_names=["Reaction"]);

# 12.3 Predictive comparison

What do PSIS-LOO diagnostics and ELPD differences say about likelihood choice and the value of varying slopes?

In [ ]:
loos = {
    name: azs.loo(idata, var_name="Reaction", pointwise=True)
    for name, idata in idatas.items()
}

for name, loo in loos.items():
    print(name, "ELPD:", round(float(loo.elpd), 2), "SE:", round(float(loo.se), 2),
          "max Pareto k:", round(float(loo.pareto_k.max()), 2))

comparison = azs.compare(loos, method="stacking", round_to="none")
comparison

In [ ]:
azp.plot_compare(comparison);

In [ ]:
azp.plot_khat(loos["gaussian_slope"]);
azp.plot_khat(loos["student_t_slope"]);

# 12.4 Prediction target

What prediction problem does this leave-one-observation-out comparison actually answer?

This comparison asks about prediction of another observation from an **already observed participant**, because rows—not whole participants—are left out. It is not a validation of prediction for a completely new participant. Compare ELPD differences together with their uncertainty and Pareto-$k$ diagnostics; do not treat a numerical rank as proof that one scientific model is true.